In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

### ---------------------------------------------------------
### Passo 1 - Ingestão de Dados
### ---------------------------------------------------------

In [2]:
print("1. Carregando base de dados...")
tabela = pd.read_csv("clientes.csv")

# Separando Target (Alvo) das Features (Características)
y = tabela["score_credito"]
X = tabela.drop(columns=["score_credito", "id_cliente"])

# Divisão Treino/Teste COM Semente Aleatória (random_state) para Reprodutibilidade
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=42)

1. Carregando base de dados...


### ---------------------------------------------------------
### Passo 2 - Engenharia de Features e Transformadores
### ---------------------------------------------------------

In [3]:
# Identificando dinamicamente quais colunas são texto e quais são números
colunas_categoricas = ["profissao", "mix_credito", "comportamento_pagamento"]
colunas_numericas = [col for col in X.columns if col not in colunas_categoricas]

# Criando o pré-processador modular (ColumnTransformer)
# Ele aplica a transformação correta na coluna correta, de forma isolada.
pre_processador = ColumnTransformer(
    transformers=[
        # Números passam pelo StandardScaler (média 0, desvio padrão 1)
        ('num', StandardScaler(), colunas_numericas),
        # Textos passam pelo OneHotEncoder (ignora categorias novas em produção para não quebrar o app)
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), colunas_categoricas)
    ])

### ---------------------------------------------------------
### Passo 3 - Construção dos Pipelines
### ---------------------------------------------------------
 O Pipeline garante que o dado de treino não vaze para o teste e que um novo cliente sofra exatamente as mesmas transformações matemáticas.

In [4]:
# Pipeline 1: Random Forest (n_jobs=-1 usa todos os núcleos do processador para paralelismo)
pipeline_rf = Pipeline(steps=[
    ('preprocessor', pre_processador),
    ('classificador', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

In [5]:
# Pipeline 2: K-Nearest Neighbors (KNN)
pipeline_knn = Pipeline(steps=[
    ('preprocessor', pre_processador),
    ('classificador', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
])

### ---------------------------------------------------------
### Passo 4 - Treinamento (Fit)
### ---------------------------------------------------------

In [6]:
print("2. Treinando os modelos (Fit)...")
pipeline_rf.fit(X_treino, y_treino)
pipeline_knn.fit(X_treino, y_treino)

2. Treinando os modelos (Fit)...


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['mes', 'idade',
                                                   'salario_anual',
                                                   'num_contas', 'num_cartoes',
                                                   'juros_emprestimo',
                                                   'num_emprestimos',
                                                   'dias_atraso',
                                                   'num_pagamentos_atrasados',
                                                   'num_verificacoes_credito',
                                                   'divida_total',
                                                   'taxa_uso_credito',
                                                   'idade_historico_credito',
                                                   'investimento_mensal',
                                                   'saldo_final_mes',
                                                   'emprestimo_carro',
                                                   'emprestimo_casa',
                                                   'emprestimo_pessoal',
                                                   'emprestimo_credito',
                                                   'emprestimo_estudantil']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['profissao', 'mix_credito',
                                                   'comportamento_pagamento'])])),
                ('classificador', KNeighborsClassifier(n_jobs=-1))])

### ---------------------------------------------------------
### Avaliação de Risco
### ---------------------------------------------------------

In [7]:
print("3. Avaliando os modelos no ambiente de Teste...\n")

previsao_rf = pipeline_rf.predict(X_teste)
previsao_knn = pipeline_knn.predict(X_teste)

print("=== RELATÓRIO DO MODELO: RANDOM FOREST ===")
# O Classification Report traz Acurácia, Precisão, Recall e F1-Score para cada classe
print(classification_report(y_teste, previsao_rf))

print("\n=== RELATÓRIO DO MODELO: KNN ===")
print(classification_report(y_teste, previsao_knn))

3. Avaliando os modelos no ambiente de Teste...

=== RELATÓRIO DO MODELO: RANDOM FOREST ===
              precision    recall  f1-score   support

        Good       0.80      0.78      0.79      5322
        Poor       0.80      0.84      0.82      8805
    Standard       0.84      0.83      0.83     15873

    accuracy                           0.82     30000
   macro avg       0.81      0.82      0.82     30000
weighted avg       0.82      0.82      0.82     30000


=== RELATÓRIO DO MODELO: KNN ===
              precision    recall  f1-score   support

        Good       0.69      0.72      0.70      5322
        Poor       0.77      0.77      0.77      8805
    Standard       0.81      0.79      0.80     15873

    accuracy                           0.77     30000
   macro avg       0.75      0.76      0.76     30000
weighted avg       0.77      0.77      0.77     30000



### ---------------------------------------------------------
### Passo 6 - Inferência em Produção (Novos Clientes)
### ---------------------------------------------------------

In [8]:
print("4. Simulando entrada de novos clientes (Produção)...")
tabela_novos_clientes = pd.read_csv("novos_clientes.csv")

nova_previsao = pipeline_rf.predict(tabela_novos_clientes)

print(f"Previsões para novos clientes: {nova_previsao}")

4. Simulando entrada de novos clientes (Produção)...
Previsões para novos clientes: ['Poor' 'Poor' 'Standard']
